# 04 · Sweep analysis and a first SINDy fit

This notebook has three parts:

1. **Sweep overview.** Condense all `5m_attack_*` datasets into one table and a few plots:
   win rates, battle durations in seconds, and which settings give repeated (deterministic) games.
2. **Level A for one dataset.** Look at the team-level trajectories of a chosen dataset,
   split by who won.
3. **First SINDy fit.** Fit a model of how team health evolves, evaluate it by simulating
   held-out episodes, and save the result as a run in `runs/`.

Settings that you may want to change are collected in the next cell.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps

from sindy_wm.data import storage
from sindy_wm.evaluation.plots import TEAM_COLORS, plot_many_episodes
from sindy_wm.paths import PROJECT_ROOT, RAW_DIR
from sindy_wm.provenance import code_info
from sindy_wm.states.aggregate import level_a

plt.rcParams["figure.dpi"] = 100

# --- Part 1: Which experiment to analyse
EXPERIMENT = "exp02_5m_random_position_aggression_stepmul"
EXPERIMENT_DIR = RAW_DIR / EXPERIMENT

# --- Part 2 and 3: the dataset to analyse and fit (pick it after looking at Part 1)
DATASET = "5m_attack_a070_s2_v1"

# --- Part 3: modelling choices
STATE_COLUMNS = ["health_blue", "health_red"]  # Level A variables used as SINDy state
TRIM_TO_ENGAGEMENT = True  # start each trajectory when the first damage is dealt
DOWNSAMPLE = 2  # keep every k-th step (2 at step_mul 1 gives 1/8 s per point)
SMOOTH_WINDOW = 9  # Savitzky-Golay window for derivatives, in (downsampled) points
TEST_FRACTION = 0.2  # share of episodes held out for evaluation
SPLIT_SEED = 0  # seed for the train/test split

RUNS_DIR = PROJECT_ROOT / "runs"

## Part 1 · Sweep overview

### 1.1 One table for all episodes

Each dataset's manifest records its parameters, and its episode index records the outcomes.
`load_all_indexes` combines them into one table with one row per episode.
Durations are converted to **seconds**, because a step means something different at each `step_mul`.

In [ ]:
def load_all_indexes(experiment_dir: Path) -> pd.DataFrame:
    # Episode indexes of all complete datasets in an experiment folder, plus their parameters
    tables = []
    for d in sorted(experiment_dir.iterdir()):
        if not (d / "manifest.json").exists():
            continue  # skips README.md and anything that isn't a dataset
        m = storage.load_manifest(d)
        if m["status"] != "complete":
            print(f"Skipping {d.name}: status {m['status']}")
            continue
        cfg = m["config"]
        index = storage.load_episode_index(d)
        tables.append(
            index.assign(
                experiment=experiment_dir.name,
                dataset=d.name,
                step_mul=cfg["env"]["step_mul"],
                aggression=cfg["policy"]["params"].get("aggression"),
                policy=cfg["policy"]["class"],
                duration_s=index["n_steps"] * cfg["env"]["seconds_per_step"],
            )
        )
    if not tables:
        raise FileNotFoundError(f"No complete datasets in {experiment_dir}")
    return pd.concat(tables, ignore_index=True)


episodes = load_all_indexes(EXPERIMENT_DIR)
print(f"{episodes['dataset'].nunique()} datasets, {len(episodes)} episodes")
episodes.head()

### 1.2 Summary per dataset

For each dataset:

- **win rate** with a 95% interval. With 100 episodes the interval is roughly ±10 percentage points
  near 50%, so differences smaller than that are not meaningful.
- **duration** in seconds.
- **distinct outcomes**: how many different (length, final health) combinations occur.
  A value of 1 means every episode is the same game repeated.

In [ ]:
def wilson_interval(wins: int, n: int, z: float = 1.96) -> tuple[float, float]:
    # 95% confidence interval for a win rate (Wilson score interval, works near 0 and 1)
    if n == 0:
        return np.nan, np.nan
    p = wins / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return max(centre - half, 0.0), min(centre + half, 1.0)


def summarize(group: pd.DataFrame) -> pd.Series:
    wins, n = int(group["battle_won"].sum()), len(group)
    low, high = wilson_interval(wins, n)
    outcome_cols = [c for c in ["n_steps", "final_hp_blue", "final_hp_red"] if c in group]
    return pd.Series(
        {
            "n": n,
            "win_rate": wins / n,
            "win_low": low,
            "win_high": high,
            "duration_mean_s": group["duration_s"].mean(),
            "duration_std_s": group["duration_s"].std(),
            "steps_mean": group["n_steps"].mean(),
            "truncated": group["truncated"].mean(),
            "distinct_outcomes": len(group[outcome_cols].drop_duplicates()),
        }
    )


summary = (
    episodes.groupby(["dataset", "step_mul", "aggression"])
    .apply(summarize, include_groups=False)
    .reset_index()
    .sort_values(["step_mul", "aggression"], ignore_index=True)
)
summary.style.format(
    {
        "win_rate": "{:.0%}",
        "win_low": "{:.0%}",
        "win_high": "{:.0%}",
        "duration_mean_s": "{:.1f}",
        "duration_std_s": "{:.1f}",
        "steps_mean": "{:.1f}",
        "truncated": "{:.0%}",
        "n": "{:.0f}",
        "distinct_outcomes": "{:.0f}",
    }
)

### 1.3 Win rate across the sweep

Each cell shows the Blue win rate and, below it, its 95% interval.
Cells marked *repeated* contain essentially one game played many times, so their
0% or 100% is a single outcome rather than a rate.

In [ ]:
def heatmap(summary: pd.DataFrame, value: str, fmt: str, title: str, cmap: str = "RdBu"):
    table = summary.pivot(index="aggression", columns="step_mul", values=value)
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(
        table,
        cmap=cmap,
        vmin=0 if value == "win_rate" else None,
        vmax=1 if value == "win_rate" else None,
        aspect="auto",
    )
    ax.set_xticks(range(len(table.columns)), table.columns)
    ax.set_yticks(range(len(table.index)), table.index)
    ax.set_xlabel("step_mul")
    ax.set_ylabel("aggression")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    return fig, ax, table


fig, ax, win_table = heatmap(summary, "win_rate", "{:.0%}", "Blue win rate")
by_cell = summary.set_index(["aggression", "step_mul"])
for i, a in enumerate(win_table.index):
    for j, s in enumerate(win_table.columns):
        row = by_cell.loc[(a, s)]
        label = f"{row.win_rate:.0%}\n[{row.win_low:.0%}, {row.win_high:.0%}]"
        if row.distinct_outcomes <= 2:
            label += "\nrepeated"
        dark = row.win_rate < 0.2 or row.win_rate > 0.8
        ax.text(j, i, label, ha="center", va="center", fontsize=8, color="white" if dark else "black")
plt.show()

**What to look for:** win rate should rise with aggression and fall with `step_mul`.
A plausible reason: with probability 1 − aggression Blue takes a random action, and that
action lasts one decision. At `step_mul` 1 it lasts 1/16 s; at `step_mul` 8 it lasts 0.5 s.

### 1.4 Battle duration in seconds

If the duration in seconds is similar at every `step_mul`, the battle itself is the same
physical event; `step_mul` then only changes how finely it is recorded and how often Blue decides.

In [ ]:
step_muls = sorted(summary["step_mul"].unique())
position = {s: k for k, s in enumerate(step_muls)}  # evenly spaced x positions

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for n, (a, group) in enumerate(summary.groupby("aggression")):
    x = group["step_mul"].map(position) + (n - 1.5) * 0.06  # small offset so error bars don't overlap
    axes[0].errorbar(
        x,
        group["duration_mean_s"],
        yerr=group["duration_std_s"],
        marker="o",
        capsize=3,
        label=f"aggression {a}",
    )
    axes[1].plot(x, group["steps_mean"], marker="o", label=f"aggression {a}")
axes[0].set(ylabel="duration (s)", title="Battle duration, mean ± std", ylim=(0, None))
axes[1].set(ylabel="steps per episode", title="Data points per episode", ylim=(0, None))
for ax in axes:
    ax.set_xticks(range(len(step_muls)), step_muls)
    ax.set_xlabel("step_mul")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 1.5 Candidate datasets for SINDy

A good first dataset has **balanced outcomes** (both teams win sometimes, so the model sees both
regimes), **varied episodes** (not one game repeated) and **many points per episode**.

In [ ]:
candidates = summary[summary["win_rate"].between(0.3, 0.7) & (summary["distinct_outcomes"] > 10)].sort_values(
    "steps_mean", ascending=False
)
candidates[["dataset", "win_rate", "steps_mean", "duration_mean_s", "distinct_outcomes"]]

Set `DATASET` in the first cell to the dataset you want to use in Parts 2 and 3,
and rerun from there.

## Part 2 · Level A trajectories of one dataset

### 2.1 Load and compute Level A

Level A is computed per episode with `normalize=True`, so health and alive counts run from 1 down to 0.
The result has a two-level index, `(episode_idx, step)`. Time in seconds is added as a column.

In [ ]:
DATASET = "5m_attack_a090_s2_v1"

dataset_dir = EXPERIMENT_DIR / DATASET
manifest = storage.load_manifest(dataset_dir)
index = storage.load_episode_index(dataset_dir).set_index("episode_idx")
dt = manifest["config"]["env"]["seconds_per_step"]

snaps = storage.load_snapshots(dataset_dir)
states = snaps.groupby("episode_idx").apply(level_a, normalize=True, include_groups=False)
states["t"] = states.index.get_level_values("step") * dt

print(f"{DATASET}: {len(index)} episodes, dt = {dt} s, Blue win rate {index['battle_won'].mean():.0%}")
states.head()

### 2.2 Won and lost episodes side by side

Each line is one episode, plotted against time in seconds.

In [ ]:
def episodes_by_time(states: pd.DataFrame, episode_ids) -> list[pd.DataFrame]:
    # One table per episode, indexed by time in seconds (for plot_many_episodes)
    return [states.loc[i].set_index("t") for i in episode_ids]


won_ids = index.index[index["battle_won"]]
lost_ids = index.index[~index["battle_won"]]

for ids, label in [(won_ids, "Blue won"), (lost_ids, "Blue lost")]:
    if len(ids) == 0:
        print(f"No episodes where {label.lower()}")
        continue
    fig = plot_many_episodes(episodes_by_time(states, ids), title=f"{DATASET}, {label}", alpha=0.2)
    fig.axes[-1].set_xlabel("time (s)")
    plt.show()

### 2.3 Phase portrait: health of Blue against health of Red

This view removes time and shows how the two teams wear each other down.
Lanchester's **square law** (each side's losses proportional to the other side's strength)
predicts curves along which $a\,B^2 - b\,R^2$ stays constant: hyperbola-like curves that end on
one of the axes. Under the **linear law** the curves would be straight lines.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for i in index.index:
    ep = states.loc[i]
    color = TEAM_COLORS["blue"] if index.loc[i, "battle_won"] else TEAM_COLORS["red"]
    ax.plot(ep["health_red"], ep["health_blue"], color=color, alpha=0.25, linewidth=1)
ax.plot([], [], color=TEAM_COLORS["blue"], label="Blue won")
ax.plot([], [], color=TEAM_COLORS["red"], label="Blue lost")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="equal health")
ax.set(
    xlabel="health_red (normalized)",
    ylabel="health_blue (normalized)",
    xlim=(0, 1.02),
    ylim=(0, 1.02),
    aspect="equal",
    title=DATASET,
)
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.show()

## Part 3 · A first SINDy fit

### 3.1 From Level A to trajectories

Choices made here (all set in the first cell):

- **State:** `STATE_COLUMNS`, by default the two team healths. The alive counts are left out because
  they are step functions, which suit derivative-based fitting poorly.
- **One trajectory per episode.** Episodes are never stacked into one array, since the jump between
  the end of one episode and the start of the next would look like a huge derivative.
- **Trim to engagement:** before the first shot, units are only walking towards each other and health
  does not change. A model that only sees health cannot tell this phase apart, so trajectories start
  one step before the first damage.
- **Downsample** to reduce the staircase of individual shots; derivatives are smoothed further with
  a Savitzky–Golay filter (`SMOOTH_WINDOW`).
- **Split by episode** into training and test episodes.

In [ ]:
def to_trajectory(ep: pd.DataFrame) -> pd.DataFrame:
    # Preprocess one episode's Level A table into the rows used for fitting
    ep = ep.reset_index(drop=True)
    if TRIM_TO_ENGAGEMENT:
        total = ep[STATE_COLUMNS].sum(axis=1)
        damaged = np.flatnonzero(total.to_numpy() < total.iloc[0] - 1e-9)
        start = max(int(damaged[0]) - 1, 0) if len(damaged) else 0
        ep = ep.iloc[start:]
    ep = ep.iloc[::DOWNSAMPLE]
    return ep.assign(t=ep["t"] - ep["t"].iloc[0])  # time from the start of the trajectory


trajectories = {i: to_trajectory(states.loc[i]) for i in index.index}

rng = np.random.default_rng(SPLIT_SEED)
episode_ids = rng.permutation(index.index.to_numpy())
n_test = int(round(TEST_FRACTION * len(episode_ids)))
test_ids = sorted(episode_ids[:n_test].tolist())
train_ids = sorted(episode_ids[n_test:].tolist())

dt_fit = dt * DOWNSAMPLE
lengths = pd.Series({i: len(trajectories[i]) for i in index.index})
print(f"{len(train_ids)} training and {len(test_ids)} test episodes, dt = {dt_fit} s")
print(f"Points per trajectory: mean {lengths.mean():.0f}, min {lengths.min()}, max {lengths.max()}")
if lengths.min() < SMOOTH_WINDOW:
    print("Warning: some trajectories are shorter than SMOOTH_WINDOW; lower it or DOWNSAMPLE.")

### 3.2 Fit

Feature library: polynomials of the state. Degree 1 can express Lanchester-type laws
($\dot B = -a R$, $\dot R = -b B$); degree 2 adds products and squares.
The optimizer (STLSQ) sets coefficients below `threshold` to zero, which keeps the model sparse.

In [ ]:
def fit_sindy(train_ids, degree: int, threshold: float) -> ps.SINDy:
    model = ps.SINDy(
        optimizer=ps.STLSQ(threshold=threshold),
        feature_library=ps.PolynomialLibrary(degree=degree),
        differentiation_method=ps.SmoothedFiniteDifference(
            smoother_kws={"window_length": SMOOTH_WINDOW, "polyorder": 2}
        ),
    )
    x = [trajectories[i][STATE_COLUMNS].to_numpy() for i in train_ids]
    model.fit(x, t=dt_fit, feature_names=STATE_COLUMNS)
    return model


model = fit_sindy(train_ids, degree=1, threshold=0.02)
model.print()

**What to look for:** under the square law, `health_blue'` depends mainly on `health_red` with a
negative coefficient, and the other way round. A constant term means losses that do not depend on
the opponent's strength, and a term in a team's own health means its losses depend on itself.

### 3.3 Evaluate by simulating test episodes

Fitting derivatives well is not the same as predicting a battle well. For each test episode, the
model is integrated forward and compared with what actually happened, in two ways:

- **From the start of the engagement.** Every episode starts from (almost) the same state, so a
  deterministic model predicts (almost) the same battle every time: it can only learn the *average*
  fight. Which side wins depends on random events during the battle, so the winner accuracy from
  the start cannot be expected to beat always guessing the more common outcome (`majority_baseline`).
  The RMSE shows how well the model captures the average course of a battle.
- **From midway.** Starting from the actual state halfway through the trajectory, where episodes
  differ, tests whether the model knows how the current balance of health decides the rest of the fight.
  Its naive baseline is `leader_baseline_midway`: predicting that whoever has more health midway wins.
  The model is only useful here if it beats that.

The predicted winner is the team with more health left at the time the real battle ended.

In [ ]:
def simulate(model: ps.SINDy, traj: pd.DataFrame, start: int = 0) -> np.ndarray | None:
    # Integrate the model from row `start` of a trajectory over the remaining time points
    t = traj["t"].to_numpy()[start:]
    x0 = traj[STATE_COLUMNS].iloc[start].to_numpy()
    try:
        sim = model.simulate(x0, t - t[0], integrator_kws={"method": "LSODA", "rtol": 1e-6, "atol": 1e-8})
    except Exception:  # integration can fail if the model blows up
        return None
    return sim if len(sim) == len(t) and np.all(np.isfinite(sim)) else None


def evaluate(model: ps.SINDy, ids) -> pd.DataFrame:
    # Metrics per test episode, simulating from the start and from midway
    blue, red = STATE_COLUMNS.index("health_blue"), STATE_COLUMNS.index("health_red")
    rows = []
    for i in ids:
        traj = trajectories[i]
        actual = traj[STATE_COLUMNS].to_numpy()
        blue_won = bool(index.loc[i, "battle_won"])
        mid = len(traj) // 2
        row = {
            "episode_idx": i,
            "blue_won": blue_won,
            # Naive baseline: whoever has more health midway wins
            "leader_correct_midway": bool((actual[mid, blue] > actual[mid, red]) == blue_won),
        }
        for label, start in [("start", 0), ("midway", mid)]:
            sim = simulate(model, traj, start)
            if sim is None:
                row |= {f"rmse_{label}": np.nan, f"winner_correct_{label}": False, f"failed_{label}": True}
                continue
            row |= {
                f"rmse_{label}": float(np.sqrt(np.mean((sim - actual[start:]) ** 2))),
                f"winner_correct_{label}": bool((sim[-1, blue] > sim[-1, red]) == blue_won),
                f"failed_{label}": False,
            }
        rows.append(row)
    return pd.DataFrame(rows).set_index("episode_idx")


def summarize_eval(model: ps.SINDy, ids) -> dict:
    ev = evaluate(model, ids)
    blue_share = ev["blue_won"].mean()
    return {
        "rmse_start": float(ev["rmse_start"].mean()),
        "rmse_midway": float(ev["rmse_midway"].mean()),
        "winner_acc_start": float(ev["winner_correct_start"].mean()),
        "winner_acc_midway": float(ev["winner_correct_midway"].mean()),
        "majority_baseline": float(max(blue_share, 1 - blue_share)),
        "leader_baseline_midway": float(ev["leader_correct_midway"].mean()),
        "failed_simulations": int(ev["failed_start"].sum() + ev["failed_midway"].sum()),
        "n_terms": int(np.count_nonzero(model.coefficients())),
    }


summarize_eval(model, test_ids)

### 3.4 Compare a few model choices

A small grid over library degree and sparsity threshold. Compare winner accuracies against their baselines,
and prefer the **simplest model that predicts about as well as the best one**: fewer terms are easier to interpret and less likely to overfit.
The comparison uses the test episodes; for the final thesis results, keep a separate set of episodes
(for example a dataset with new seeds) that is never used for choosing models.

In [ ]:
comparison = []
for degree in [1, 2]:
    for threshold in [0.005, 0.02, 0.05, 0.1]:
        m = fit_sindy(train_ids, degree, threshold)
        comparison.append({"degree": degree, "threshold": threshold, **summarize_eval(m, test_ids)})
comparison = pd.DataFrame(comparison)
comparison

Set the chosen model below. Then look at a few test episodes: actual (solid), simulated from the start (dashed)
and from midway (dotted).

In [ ]:
DEGREE, THRESHOLD = 2, 0.005  # chosen from the comparison above

model = fit_sindy(train_ids, DEGREE, THRESHOLD)
model.print()
metrics = summarize_eval(model, test_ids)
per_episode = evaluate(model, test_ids)
metrics

In [ ]:
def plot_predictions(model, ids, title=""):
    fig, axes = plt.subplots(1, len(ids), figsize=(4 * len(ids), 3.5), sharey=True, squeeze=False)
    for ax, i in zip(axes[0], ids, strict=True):
        traj = trajectories[i]
        mid = len(traj) // 2
        sim_start, sim_mid = simulate(model, traj), simulate(model, traj, mid)
        for k, col in enumerate(STATE_COLUMNS):
            color = TEAM_COLORS.get(col.split("_")[-1], "k")
            ax.plot(traj["t"], traj[col], color=color, label=f"{col} actual")
            if sim_start is not None:
                ax.plot(traj["t"], sim_start[:, k], "--", color=color, label="model from start")
            if sim_mid is not None:
                ax.plot(
                    traj["t"].iloc[mid:],
                    sim_mid[:, k],
                    ":",
                    color=color,
                    linewidth=2,
                    label="model from midway",
                )
        outcome = "Blue won" if index.loc[i, "battle_won"] else "Blue lost"
        ax.set(title=f"episode {i} ({outcome})", xlabel="time since engagement (s)")
        ax.grid(True, alpha=0.3)
    axes[0][0].set_ylabel("health (normalized)")
    axes[0][-1].legend(fontsize=7)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


test_won = [i for i in test_ids if index.loc[i, "battle_won"]][:2]
test_lost = [i for i in test_ids if not index.loc[i, "battle_won"]][:2]
fig_predictions = plot_predictions(model, test_won + test_lost, title=f"{DATASET}: test episodes")
plt.show()

### 3.5 Save as a run

Everything needed to understand and redo this result goes into one new folder in `runs/`:

| file | contents |
|---|---|
| `config.json` | dataset (name and creation time), train/test episodes, preprocessing and model choices, code version |
| `model.json` | state and feature names, coefficients, equations |
| `equations.txt` | the model in readable form |
| `metrics.json` | summary metrics on the test episodes |
| `per_episode.csv` | metrics per test episode |
| `figures/` | the prediction plot |

An existing run folder is never overwritten. `save_run` is defined here as a prototype; following
the project conventions (code that produces saved data belongs in `src/`), it should move to
`src/sindy_wm/` once the format settles.

In [ ]:
def save_run(
    name: str, config: dict, model: ps.SINDy, metrics: dict, per_episode: pd.DataFrame, figures: dict
) -> Path:
    run_dir = RUNS_DIR / f"{datetime.now():%Y-%m-%d_%H%M}_{name}"
    run_dir.mkdir(parents=True, exist_ok=False)  # never overwrite a run
    (run_dir / "figures").mkdir()

    def write_json(filename, obj):
        (run_dir / filename).write_text(json.dumps(obj, indent=2, default=str), encoding="utf-8")

    write_json("config.json", {**config, "code": code_info()})
    write_json(
        "model.json",
        {
            "state_names": STATE_COLUMNS,
            "feature_names": model.get_feature_names(),
            "coefficients": model.coefficients().tolist(),  # one row per state variable
            "equations": model.equations(precision=5),
        },
    )
    (run_dir / "equations.txt").write_text(
        "\n".join(f"({s})' = {eq}" for s, eq in zip(STATE_COLUMNS, model.equations(precision=5), strict=True))
        + "\n",
        encoding="utf-8",
    )
    write_json("metrics.json", metrics)
    per_episode.to_csv(run_dir / "per_episode.csv")
    for filename, fig in figures.items():
        fig.savefig(run_dir / "figures" / filename, dpi=150, bbox_inches="tight")
    return run_dir


run_config = {
    "description": "First SINDy fit on Level A team health",
    "dataset": {"name": DATASET, "created": manifest["created"]},
    "split": {
        "train": [int(i) for i in train_ids],
        "test": [int(i) for i in test_ids],
        "seed": SPLIT_SEED,
        "test_fraction": TEST_FRACTION,
    },
    "preprocessing": {
        "state_columns": STATE_COLUMNS,
        "normalize": True,
        "trim_to_engagement": TRIM_TO_ENGAGEMENT,
        "downsample": DOWNSAMPLE,
        "dt": dt_fit,
    },
    "model": {
        "library": "PolynomialLibrary",
        "degree": DEGREE,
        "optimizer": "STLSQ",
        "threshold": THRESHOLD,
        "differentiation": "SmoothedFiniteDifference (Savitzky-Golay)",
        "smooth_window": SMOOTH_WINDOW,
        "pysindy": ps.__version__,
    },
}

run_dir = save_run(
    name=f"levelA_health_poly{DEGREE}",
    config=run_config,
    model=model,
    metrics=metrics,
    per_episode=per_episode,
    figures={"test_predictions.png": fig_predictions},
)
print(f"Saved run to {run_dir}")
print(sorted(p.relative_to(run_dir).as_posix() for p in run_dir.rglob("*")))

## Next steps

- Write the main findings from Part 1 into the README's findings section.
- Try other state choices, e.g. adding `distance`, or a model per phase (approach, then fight).
- Check how sensitive the fit is to `DOWNSAMPLE` and `SMOOTH_WINDOW`.
- Fit on other balanced datasets from Part 1 and compare the coefficients: do they change
  systematically with aggression?